# Icosahedral Global Graph - Using `weather-model-graphs`

This notebook walks through the `weather-model-graphs` API for creating global spherical graphs based on icosahedral meshes.  We visualise the outputs at each step so you can build up an intuition for what the library is constructing.

1. **Mesh hierarchy** - generate icosahedral meshes at increasing refinement levels
2. **Input grid** - define the lat/lon grid that provides the input features
3. **Grid-to-mesh (G2M) connectivity** - connect grid points to the finest mesh
4. **Mesh graphs** - flat and hierarchical mesh-to-mesh connectivity
5. **End-to-end with `create_all_graph_components`** - flat and hierarchical graphs
6. **Visualise graph components** - inspect the m2m, g2m and m2g sub-graphs

For a deep-dive into how the library builds these graphs internally (trimesh subdivision, face-adjacency, barycentric interpolation) see the companion notebook `docs/internals/icosahedral_global_graph_internals.ipynb`.

In [ ]:
!pip install loguru trimesh rtree pyproj networkx scipy plotly pandas torch torch-geometric

In [ ]:
!pip install nbformat>=4.2.0 ipykernel 

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected" 
# ou "vscode" si tu utilises l'éditeur VS Code

In [ ]:
import numpy as np
import pyproj
import plotly.graph_objects as go

In [ ]:
import sys
import os
from pathlib import Path

# We define the path to the 'src' folder of the project
# Structure : /docs/internals/notebook.ipynb -> /src/
current_dir = Path.cwd()
root_dir = current_dir.parent.parent
src_path = str(root_dir / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)


# Test de présence du module
try:
    import weather_model_graphs
    print(" Success: The weather_model_graphs module is now accessible!")
except ImportError as e:
    print(f"❌ faild : {e}")

In [ ]:
from weather_model_graphs.create.mesh.layouts.icosahedral import (
    create_hierarchy_of_icosahedral_meshes,
    create_flat_icosahedral_mesh_graph,
    create_hierarchical_icosahedral_mesh_graph,
    connect_grid_to_mesh,
    cartesian_to_lat_lon,
)
from weather_model_graphs.create import create_all_graph_components

## Helper: `plot_graph_on_globe`

A small reusable Plotly helper that renders any set of (lat, lon) nodes and edges on an equirectangular map.  We define it once here and call it after each construction step.

In [ ]:
def plot_graph_on_globe(
    node_lat,
    node_lon,
    *,
    edge_pairs=None,  # list of (i, j) index pairs into node arrays
    node_color="steelblue",
    node_size=4,
    title="",
    width=900,
    height=500,
):
    """Lightweight Plotly visualisation of a graph in geographic coordinates.

    Parameters
    ----------
    node_lat, node_lon : array-like
        Node coordinates in degrees.
    edge_pairs : list of (int, int), optional
        Index pairs.  Edges that cross the antimeridian (|Δlon| > 180) are
        skipped to avoid wrap-around artefacts.
    """
    edge_lons, edge_lats = [], []
    if edge_pairs is not None:
        for i, j in edge_pairs:
            if abs(float(node_lon[i]) - float(node_lon[j])) < 180:
                edge_lons += [node_lon[i], node_lon[j], None]
                edge_lats += [node_lat[i], node_lat[j], None]

    traces = []
    if edge_lons:
        traces.append(
            go.Scatter(
                x=edge_lons,
                y=edge_lats,
                mode="lines",
                line=dict(color="lightsteelblue", width=0.8),
                hoverinfo="skip",
                showlegend=False,
            )
        )
    traces.append(
        go.Scatter(
            x=node_lon,
            y=node_lat,
            mode="markers",
            marker=dict(
                size=node_size, color=node_color, line=dict(color="white", width=0.3)
            ),
            hovertemplate="lat=%{y:.2f}°, lon=%{x:.2f}°<extra></extra>",
        )
    )

    fig = go.Figure(traces)
    fig.update_layout(
        title=title,
        showlegend=False,
        xaxis=dict(
            title="Longitude",
            range=[-185, 185],
            dtick=30,
            showgrid=True,
            gridcolor="#eee",
        ),
        yaxis=dict(
            title="Latitude", range=[-95, 95], dtick=30, showgrid=True, gridcolor="#eee"
        ),
        width=width,
        height=height,
        margin=dict(l=40, r=20, t=50, b=40),
        plot_bgcolor="white",
    )
    return fig

## 1. Generate the Icosahedral Mesh Hierarchy

`create_hierarchy_of_icosahedral_meshes` returns a list of `(vertices, faces)` tuples, one per refinement level.  Each successive level subdivides every triangle into four, so the node count grows roughly 4× per level.

In [ ]:
max_subdivisions = 3
mesh_list = create_hierarchy_of_icosahedral_meshes(max_subdivisions)

print("Level │   Nodes │   Faces")
for level, (verts, faces) in enumerate(mesh_list):
    print(f"  {level}   │  {len(verts):5d}  │  {len(faces):5d}")

Each level on its own looks like a progressively finer quasi-uniform point cloud on the sphere.  The plot below shows the finest level (`subdivisions=3`).

In [ ]:
finest_verts, finest_faces = mesh_list[-1]
ll = cartesian_to_lat_lon(finest_verts)

# Build edge pairs from triangle faces
edge_set = set()
for f in finest_faces:
    for a, b in [(0, 1), (1, 2), (2, 0)]:
        edge_set.add((min(f[a], f[b]), max(f[a], f[b])))

fig = plot_graph_on_globe(
    ll[:, 0],
    ll[:, 1],
    edge_pairs=list(edge_set),
    node_size=3,
    title=f"Icosahedral mesh - level {max_subdivisions}  "
    f"({len(finest_verts)} nodes, {len(finest_faces)} faces)",
)
fig.show()

## 2. Define the Input Grid

We use a regular 5° lat/lon grid as the input feature grid.  A coarser resolution is intentional here, it keeps the visualisations readable.  In practice you would use whatever grid your NWP model outputs.

In [ ]:
resolution = 5.0  # degrees  (use 2.5 for production)
lats = np.arange(-90, 90.1, resolution)
lons = np.arange(-180, 180.1, resolution)
grid_lat, grid_lon = np.meshgrid(lats, lons, indexing="ij")
grid_lat_lon = np.column_stack([grid_lat.ravel(), grid_lon.ravel()])

print(f"Grid: {len(lats)} x {len(lons)} = {len(grid_lat_lon)} nodes")

fig = plot_graph_on_globe(
    grid_lat_lon[:, 0],
    grid_lat_lon[:, 1],
    node_color="tomato",
    node_size=3,
    title=f"Input grid = {resolution}deg resolution ({len(grid_lat_lon)} nodes)",
)
fig.show()

## 3. Connect Grid to Mesh (G2M)

`connect_grid_to_mesh` links every grid point to the nearby mesh nodes of the finest level using a radius search.  `radius_factor=0.6` means the search radius is 0.6× the maximum edge length of the mesh - large enough to guarantee every grid point gets at least one connection.

In [ ]:
g2m_edges = connect_grid_to_mesh(
    grid_lat_lon,
    finest_verts,
    finest_faces,
    radius_factor=0.6,
)

print(f"G2M edges:  {g2m_edges.shape[1]}")
print(f"Avg connections per grid node: {g2m_edges.shape[1]/len(grid_lat_lon):.2f}")
print(f"Avg connections per mesh node: {g2m_edges.shape[1]/len(finest_verts):.2f}")

In [ ]:
# Visualise: blue = mesh nodes, red = grid nodes, grey lines = G2M edges
mesh_ll = cartesian_to_lat_lon(finest_verts)

edge_pairs_g2m = list(zip(g2m_edges[0].tolist(), g2m_edges[1].tolist()))

# Build combined node arrays: mesh nodes first, then grid nodes
all_lat = np.concatenate([mesh_ll[:, 0], grid_lat_lon[:, 0]])
all_lon = np.concatenate([mesh_ll[:, 1], grid_lat_lon[:, 1]])
n_mesh = len(mesh_ll)

# Offset grid indices by n_mesh so edge pairs index into all_lat/all_lon
offset_pairs = [(n_mesh + gi, mi) for gi, mi in edge_pairs_g2m]

fig = plot_graph_on_globe(
    all_lat,
    all_lon,
    edge_pairs=offset_pairs[:2000],  # subsample for legibility
    node_color=["steelblue"] * n_mesh + ["tomato"] * len(grid_lat_lon),
    node_size=3,
    title="G2M connectivity : grey lines connect grid points to mesh nodes",
)
fig.show()

## 4. Mesh Graphs

Before using the end-to-end API it is helpful to look at the mesh graph on its own.  `create_flat_icosahedral_mesh_graph` returns a single-level graph; `create_hierarchical_icosahedral_mesh_graph` stacks multiple levels with inter-level edges.

In [ ]:
G_flat = create_flat_icosahedral_mesh_graph(subdivisions=3)
print(f"Flat mesh : nodes: {len(G_flat.nodes)}, edges: {len(G_flat.edges)}")

G_hier = create_hierarchical_icosahedral_mesh_graph(max_subdivisions=3)
print(f"Hier mesh : nodes: {len(G_hier.nodes)}, edges: {len(G_hier.edges)}")

level_counts = {}
for node, data in G_hier.nodes(data=True):
    l = data.get("level", 0)
    level_counts[l] = level_counts.get(l, 0) + 1
print("Hierarchical nodes per level:", dict(sorted(level_counts.items())))

## 5. End-to-End: `create_all_graph_components`

`create_all_graph_components` builds the full G2M + M2M + M2G graph in one call.  Below we create both a flat and a hierarchical variant.

In [ ]:
geographic_crs = pyproj.CRS.from_string("EPSG:4326")

### 5a. Flat mesh

In [ ]:
G_complete = create_all_graph_components(
    coords=grid_lat_lon,
    mesh_layout="icosahedral",
    m2m_connectivity="icosahedral",
    m2m_connectivity_kwargs={"subdivisions": 3, "hierarchical": False, "radius": 1.0},
    g2m_connectivity="within_radius",
    m2g_connectivity="within_radius",
    g2m_connectivity_kwargs={"rel_max_dist": 0.6},
    m2g_connectivity_kwargs={"rel_max_dist": 0.6},
    coords_crs=geographic_crs,
    graph_crs=geographic_crs,
)

node_types = {}
edge_comps = {}
for _, d in G_complete.nodes(data=True):
    node_types[d.get("type", "?")] = node_types.get(d.get("type", "?"), 0) + 1
for _, _, d in G_complete.edges(data=True):
    edge_comps[d.get("component", "?")] = edge_comps.get(d.get("component", "?"), 0) + 1
print("Nodes:", node_types)
print("Edges:", edge_comps)

### 5b. Hierarchical mesh

In [ ]:
G_hier_complete = create_all_graph_components(
    coords=grid_lat_lon,
    mesh_layout="icosahedral",
    m2m_connectivity="icosahedral",
    m2m_connectivity_kwargs={
        "max_subdivisions": 3,
        "hierarchical": True,
        "radius": 1.0,
    },
    g2m_connectivity="within_radius",
    m2g_connectivity="within_radius",
    g2m_connectivity_kwargs={"rel_max_dist": 0.6},
    m2g_connectivity_kwargs={"rel_max_dist": 0.6},
    coords_crs=geographic_crs,
    graph_crs=geographic_crs,
)

mesh_levels = {}
for _, d in G_hier_complete.nodes(data=True):
    if d.get("type") == "mesh":
        l = d.get("level", 0)
        mesh_levels[l] = mesh_levels.get(l, 0) + 1
print("Hierarchical mesh nodes per level:", dict(sorted(mesh_levels.items())))

## 6. Visualise Graph Components

The helper below extracts node positions from a `weather-model-graphs` graph object (handling both `pos3d` Cartesian and plain lat/lon node attributes) and returns arrays suitable for `plot_graph_on_globe`.

In [ ]:
def extract_component(G, component_name):
    """Return (node_lat, node_lon, edge_pairs) for a named graph component."""
    edges = [
        (u, v) for u, v, d in G.edges(data=True) if d.get("component") == component_name
    ]
    if not edges:
        raise ValueError(f"No edges with component='{component_name}'")
    G_sub = G.edge_subgraph(edges)

    index = {}  # node → position in output arrays
    lats, lons, colors = [], [], []

    for node in G_sub.nodes:
        d = G.nodes[node]
        if "pos3d" in d and d["pos3d"] is not None:
            ll = cartesian_to_lat_lon(np.asarray(d["pos3d"]).reshape(1, 3))[0]
            lat, lon = ll[0], ll[1]
        elif "pos" in d and d["pos"] is not None:
            lat, lon = d["pos"][0], d["pos"][1]
        elif "lat" in d and "lon" in d:
            lat, lon = d["lat"], d["lon"]
        else:
            continue
        index[node] = len(lats)
        lats.append(lat)
        lons.append(lon)
        colors.append("steelblue" if d.get("type") == "mesh" else "tomato")

    pairs = [(index[u], index[v]) for u, v in edges if u in index and v in index]
    return np.array(lats), np.array(lons), colors, pairs

### M2M : mesh-to-mesh connectivity

The m2m edges connect each mesh node to its face-adjacent neighbours on the same level (flat mesh) or to neighbours on adjacent levels (hierarchical).  The quasi-uniform coverage of the icosahedron is clearly visible.

In [ ]:
lat_, lon_, col_, pairs_ = extract_component(G_complete, "m2m")
fig = plot_graph_on_globe(
    lat_,
    lon_,
    edge_pairs=pairs_,
    node_size=4,
    title="M2M — mesh-to-mesh connectivity (flat, subdivisions=3)",
)
fig.show()

### G2M : grid-to-mesh connectivity

Each grid point (red) fans out to the nearby mesh nodes (blue) within the search radius.  This is the first message-passing step in a GNN: information flows from the input grid onto the mesh.

In [ ]:
lat_, lon_, col_, pairs_ = extract_component(G_complete, "g2m")
fig = plot_graph_on_globe(
    lat_,
    lon_,
    edge_pairs=pairs_[:3000],
    node_color=col_,
    node_size=4,
    title="G2M - grid-to-mesh connectivity (subsampled edges)",
)
fig.show()

### M2G : mesh-to-grid connectivity

The reverse direction: each output grid point (red) receives information from the surrounding mesh nodes (blue).  Edge density mirrors the G2M graph.

In [ ]:
lat_, lon_, col_, pairs_ = extract_component(G_complete, "m2g")
fig = plot_graph_on_globe(
    lat_,
    lon_,
    edge_pairs=pairs_[:3000],
    node_color=col_,
    node_size=4,
    title="M2G - mesh-to-grid connectivity (subsampled edges)",
)
fig.show()

In [ ]:
import numpy as np
import json
import os
import plotly.graph_objects as go
import plotly.io as pio

# 1. SETUP: Force browser rendering for Ubuntu/VS Code stability
pio.renderers.default = "browser"

def generate_graph_metadata(output_path, mesh_levels=3):
    """
    Addresses Gap 6: Generates the metadata.json spec required by 
    the Neural-LAM validator for spherical/global topologies.
    """
    metadata = {
        "graph_type": "icosahedral_global",
        "coordinate_system": "3d_cartesian",
        "spatial_units": "unit_sphere_normalized",
        "hierarchy": {
            "levels": mesh_levels,
            "is_hierarchical": True,
            "inter_level_edges": "present" # Mapping L(i) to L(i+1)
        },
        "edge_features": {
            "m2m": ["distance", "tan_dx", "tan_dy"],
            "g2m": ["barycentric_weights", "spherical_distance"]
        },
        "notes": "Compliant with Neural-LAM 3D Cartesian validation path"
    }
    
    file_path = os.path.join(output_path, "metadata.json")
    with open(file_path, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"✅ Metadata spec generated: {file_path}")
    return metadata

def add_3d_cartesian_coords(G):
    """
    Projects Lat/Lon 'pos' attributes into 3D (x, y, z).
    Essential for bypassing rectilinear pole singularities.
    """
    updated_count = 0
    for n, d in G.nodes(data=True):
        pos_data = d.get('pos')
        if pos_data is not None and len(pos_data) >= 2:
            lat, lon = pos_data[0], pos_data[1]
            
            # Convert to Radians
            phi = np.deg2rad(90 - lat)
            theta = np.deg2rad(lon)
            
            # Spherical to Cartesian
            x = np.sin(phi) * np.cos(theta)
            y = np.sin(phi) * np.sin(theta)
            z = np.cos(phi)
            
            G.nodes[n]['pos3d'] = [x, y, z]
            updated_count += 1
    
    print(f"✅ Projected {updated_count} nodes to 3D Cartesian space.")
    return G

def plot_global_mesh_3d(G, title="Global Icosahedral Mesh Validation"):
    """
    Addresses Gap 3: Interactive visualization for M2G/G2M validation.
    """
    mesh_nodes = []
    grid_nodes = []
    
    for n, d in G.nodes(data=True):
        if 'pos3d' in d:
            if d.get('type') == 'mesh':
                mesh_nodes.append(d['pos3d'])
            else:
                grid_nodes.append(d['pos3d'])
                
    m_pos = np.array(mesh_nodes)
    g_pos = np.array(grid_nodes)
    
    fig = go.Figure()

    # Mesh Layer (The Icosahedron structure)
    if len(m_pos) > 0:
        fig.add_trace(go.Scatter3d(
            x=m_pos[:, 0], y=m_pos[:, 1], z=m_pos[:, 2],
            mode='markers',
            marker=dict(size=2, color='cyan', opacity=0.6),
            name="Latent Mesh (Nodes)"
        ))

    # Grid Layer (The Meteorological Input)
    if len(g_pos) > 0:
        fig.add_trace(go.Scatter3d(
            x=g_pos[:, 0], y=g_pos[:, 1], z=g_pos[:, 2],
            mode='markers',
            marker=dict(size=3, color='orange', opacity=0.8),
            name="Input Grid"
        ))

    fig.update_layout(
        template="plotly_dark",
        scene=dict(
            xaxis_title='X (Cartesian)',
            yaxis_title='Y (Cartesian)',
            zaxis_title='Z (Cartesian)',
            aspectmode='data'
        ),
        title=title
    )
    
    print(" Opening 3D Visualization in browser...")
    fig.show()

# --- EXECUTION ---
# 1. Generate the spec file
meta = generate_graph_metadata(output_path="./")

# 2. Prepare the graph
G_complete = add_3d_cartesian_coords(G_complete)

# 3. Visualize
plot_global_mesh_3d(G_complete)

## 7. Save and Load the Graph

In [ ]:
import pickle, os

os.makedirs("saved_graphs", exist_ok=True)
with open("saved_graphs/icosahedral_flat.pkl", "wb") as f:
    pickle.dump(G_complete, f)
print("Saved to saved_graphs/icosahedral_flat.pkl")

with open("saved_graphs/icosahedral_flat.pkl", "rb") as f:
    G_loaded = pickle.load(f)
print(f"Loaded => nodes: {len(G_loaded.nodes)}, edges: {len(G_loaded.edges)}")